# Held-out test (frozen config)

One-shot evaluation. Dev (`eval/golden.jsonl`, 30 q) is a sanity check against the v4 results; test (`eval/test.jsonl`, 20 q) is evaluated only after the dev numbers match. Nothing is tuned here.

In [1]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

ROOT = Path.cwd() if (Path.cwd() / "corpus").exists() else Path.cwd().parent
sys.path.append(str(ROOT / "scripts"))
from bm25 import BM25
from chunking import chunk_words
from fusion import rrf
from golden import facts, load_golden, recall_at_k
from textproc import tokenize

# Frozen configuration, do not change after seeing test results
SIZE, OVERLAP = 192, 48
DEPTH = 50        # candidates each retriever passes to RRF
RRF_TOP = 10      # length of the fused list

MODELS = {
    "e5": ("intfloat/multilingual-e5-base", "passage: "),
    "tr": ("trmteb/turkish-embedding-model", ""),
}
QUERY_PREFIX = {"e5": "query: ", "tr": ""}
HYBRIDS = {
    "hyb-e5+bm25": ["e5", "bm25-prefix5"],
    "hyb-e5+tr+bm25": ["e5", "tr", "bm25-prefix5"],
}
SYSTEMS = ["e5", "hyb-e5+bm25", "hyb-e5+tr+bm25"]
KS = (1, 3, 5, 10)

dev = load_golden(ROOT / "eval/golden.jsonl")
test = load_golden(ROOT / "eval/test.jsonl")
print(len(dev), "dev questions,", len(test), "test questions")

30 dev questions, 20 test questions


In [3]:
docs = {p.stem: p.read_text(encoding="utf-8") for p in sorted((ROOT / "corpus/text").glob("*.txt"))}
# chunk each document separately so no chunk mixes two documents
chunks = [c for t in docs.values() for c in chunk_words(t, SIZE, OVERLAP)]
print(len(docs), "documents,", len(chunks), "chunks")

models = {k: SentenceTransformer(name) for k, (name, _) in MODELS.items()}

def embed(k, texts, is_query):
    prefix = QUERY_PREFIX[k] if is_query else MODELS[k][1]
    return models[k].encode([prefix + t for t in texts], normalize_embeddings=True,
                            batch_size=32, show_progress_bar=False)

chunk_vecs = {k: embed(k, chunks, is_query=False) for k in MODELS}
bm = BM25([tokenize(c, "prefix5") for c in chunks])

5 documents, 153 chunks


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [4]:
def retrieve(questions):
    """rankings[system][question_id] = chunk texts, best first (embedding/BM25: DEPTH, hybrids: RRF_TOP)."""
    base = {}
    for k in MODELS:
        q_vecs = embed(k, [g["question"] for g in questions], is_query=True)
        scores = q_vecs @ chunk_vecs[k].T          # cosine similarity, vectors are normalized
        order = np.argsort(-scores, axis=1)[:, :DEPTH]
        base[k] = {g["id"]: [chunks[i] for i in order[n]] for n, g in enumerate(questions)}
    base["bm25-prefix5"] = {
        g["id"]: [chunks[i] for i, _ in bm.search(tokenize(g["question"], "prefix5"), k=DEPTH)]
        for g in questions
    }
    for name, parts in HYBRIDS.items():
        base[name] = {g["id"]: rrf([base[p][g["id"]] for p in parts], top=RRF_TOP) for g in questions}
    return {s: base[s] for s in SYSTEMS}

def per_question(questions, rankings):
    """DataFrame: one row per question, columns '<system> R@k'."""
    rows = []
    for g in questions:
        row = {"id": g["id"], "type": g["type"], "question": g["question"][:45]}
        for s in SYSTEMS:
            for K in KS:
                row[f"{s} R@{K}"] = recall_at_k(rankings[s][g["id"]], facts(g), K)
        rows.append(row)
    return pd.DataFrame(rows)

def summary(pq, set_name):
    return pd.DataFrame([
        {"set": set_name, "model": s, **{f"R@{K}": round(pq[f"{s} R@{K}"].mean(), 2) for K in KS}}
        for s in SYSTEMS
    ])

In [5]:
# Dev sanity check: must reproduce the v4 results at 192/48 exactly, otherwise stop before touching test
dev_rank = retrieve(dev)
dev_pq = per_question(dev, dev_rank)
dev_sum = summary(dev_pq, "dev")

EXPECTED = {
    "e5":             (0.22, 0.48, 0.52, 0.72),
    "hyb-e5+bm25":    (0.33, 0.50, 0.57, 0.65),
    "hyb-e5+tr+bm25": (0.33, 0.57, 0.62, 0.72),
}
for _, r in dev_sum.iterrows():
    got = tuple(r[f"R@{K}"] for K in KS)
    assert np.allclose(got, EXPECTED[r["model"]]), f"dev mismatch for {r['model']}: got {got}, expected {EXPECTED[r['model']]}"
print("dev matches v4")
dev_sum

dev matches v4


,set,model,R@1,R@3,R@5,R@10
0,dev,e5,0.22,0.48,0.52,0.72
1,dev,hyb-e5+bm25,0.33,0.50,0.57,0.65
2,dev,hyb-e5+tr+bm25,0.33,0.57,0.62,0.72


In [6]:
# One-shot test evaluation with the frozen configuration
test_rank = retrieve(test)
test_pq = per_question(test, test_rank)
test_sum = summary(test_pq, "test")
test_sum

,set,model,R@1,R@3,R@5,R@10
0,test,e5,0.45,0.50,0.70,0.75
1,test,hyb-e5+bm25,0.45,0.65,0.75,0.85
2,test,hyb-e5+tr+bm25,0.45,0.55,0.60,0.75


In [7]:
# Dev and test side by side
both = dev_sum.drop(columns="set").merge(test_sum.drop(columns="set"), on="model", suffixes=(" dev", " test"))
display(both)

# questions where no system has any of the evidence in its top 10 (R@10 == 0 for every system)
never = [r["id"] for _, r in test_pq.iterrows() if all(r[f"{s} R@10"] == 0 for s in SYSTEMS)]
print("test questions no system found:", never)

out = ROOT / "results"
out.mkdir(exist_ok=True)
pd.concat([dev_sum, test_sum]).to_csv(out / "test_v1_summary.csv", index=False)
pd.concat([dev_pq.assign(set="dev"), test_pq.assign(set="test")]).round(2).to_csv(out / "test_v1_per_question.csv", index=False)
print("saved")

,model,R@1 dev,R@3 dev,R@5 dev,R@10 dev,R@1 test,R@3 test,R@5 test,R@10 test
0,e5,0.22,0.48,0.52,0.72,0.45,0.50,0.70,0.75
1,hyb-e5+bm25,0.33,0.50,0.57,0.65,0.45,0.65,0.75,0.85
2,hyb-e5+tr+bm25,0.33,0.57,0.62,0.72,0.45,0.55,0.60,0.75


test questions no system found: ['q033', 'q044']
saved
